# PyTorch: ViT Paper Replicating

Link: https://arxiv.org/abs/2010.11929

In [ ]:
import torch
from torchinfo import summary
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torchmetrics.classification import MulticlassAccuracy
from common import CV_DATASETS_DIR
import common.torch as ct

In [ ]:
ct.set_default_seed()
ct.set_default_optimizations()
device = ct.get_optimal_device()

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
N_EPOCHS = 15
# Other parameters
IMAGE_SIZE = (224,224)

## Prepare Datasets

In [ ]:
DATASET_PATH = CV_DATASETS_DIR/"animals"

tr_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=v2.Compose([
                                        v2.Resize(size=IMAGE_SIZE),
                                        v2.PILToTensor(),
                                        v2.ToDtype(dtype=torch.float, scale=True),
                                    ]),
                                    split="trainval",
                                    download=True)
ts_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=v2.Compose([
                                        v2.Resize(size=IMAGE_SIZE),
                                        v2.PILToTensor(),
                                        v2.ToDtype(dtype=torch.float, scale=True),
                                    ]),
                                    split="test",
                                    download=True)

assert tr_dataset.class_to_idx == ts_dataset.class_to_idx, "Train and Test class indices mismatch!"
len(tr_dataset), len(ts_dataset)

In [ ]:
tr_dl = DataLoader(tr_dataset, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
ts_dl = DataLoader(ts_dataset, batch_size=BATCH_SIZE, num_workers=2)
len(tr_dl), len(ts_dl)

In [ ]:
n_classes = len(tr_dataset.classes)
n_classes

## Define Model

* **Inputs** – image tensors
* **Outputs**  image classifications labels
* **Layers** - Takes an input, passes it through a series of layers, and produces an output.

In [ ]:
H = 224
W = 224
C = 3
P = 16
D = P*P*C
N = H*W // P**2

 ### Image embedding

* Input shape: $(H, W, C)$, where $H$ - image height, $W$ - image width, $C$ - image number of channels
* Output shape: ${N \times\left(P^{2} \cdot C\right)}$, where $P$ - patch size, $C$ - image number of channels, $N$ - sequence length
* $N = HW / P^{2}$

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, d_model: int, patch_size: int, in_channels: int):
        super().__init__()
        self.patch_size = patch_size
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=d_model,
            kernel_size=patch_size,
            stride=patch_size,
            padding=0)
        self.flatten = nn.Flatten(start_dim=2)

    def forward(self, x: torch.Tensor):
        assert len(x.shape) == 4, "Batch dimension is absent"
        assert x.shape[-1] % self.patch_size == 0, "Image size is not divisible by patch size"
        # Project image to patches: (B, C, H, W) -> (B, D, Hp, Wp)
        x = self.conv(x)
        # Flatten the spatial dimensions: (B, D, Hp, Wp) -> (B, D, N)
        x = self.flatten(x)
        # Swap dimensions to match ViT input: (B, N, D)
        x = x.permute(0, 2, 1)
        return  x

In [ ]:
patcher = PatchEmbedding(d_model=D, patch_size=P, in_channels=C)
image_embeddings = patcher(torch.randn(1, C, H, W))
image_embeddings.shape

### Class token embedding

In [ ]:
class_embedding = nn.Parameter(torch.randn(1, 1, D), requires_grad=True)
class_embedding.shape

In [ ]:
image_and_class_embeddings = torch.cat((class_embedding, image_embeddings), dim=1)
image_and_class_embeddings.shape

### Position embedding

In [ ]:
position_embeddings = nn.Parameter(torch.randn(1, N+1, D), requires_grad=True)
position_embeddings.shape

In [ ]:
all_embeddings = torch.add(image_and_class_embeddings, position_embeddings)
all_embeddings.shape

### Multihead Self-Attention (MSA)

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(embed_dim=d_model,
                                          num_heads=n_heads,
                                          dropout=dropout,
                                          batch_first=True)

    def forward(self, x: torch.Tensor):
        x = self.norm(x)
        x, _ = self.attn(query=x, key=x, value=x, attn_mask=None)
        return x

In [ ]:
msa = MultiHeadSelfAttention(D, 12)
msa_output = msa(all_embeddings)
msa_output.shape

### Multilayer Perceptron (MLP)

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_model: int, mlp_size: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_size, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor):
        x = self.norm(x)
        x = self.mlp(x)
        return x

In [ ]:
mlp = MLP(D, mlp_size=3072)
mlp_output = mlp(msa_output)
mlp_output.shape

### Encoder

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self,
                 d_model: int,
                 n_heads: int,
                 mlp_size: int,
                 atn_dropout: float = 0.0,
                 mlp_dropout: float = 0.1):
        super().__init__()
        self.msa = MultiHeadSelfAttention(d_model, n_heads, atn_dropout)
        self.mlp = MLP(d_model, mlp_size, mlp_dropout)

    def forward(self, x: torch.Tensor):
        x = self.msa(x) + x
        x = self.mlp(x) + x
        return x

In [ ]:
encoder = TransformerEncoder(D, 12, 3072)
encoder_output = encoder(all_embeddings)
encoder_output.shape

In [ ]:
summary(model=encoder,
        input_size=(1, N+1, D),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

### ViT module

In [ ]:
class ViT(nn.Module):
    def __init__(self,
                 d_model: int,
                 image_size: int,
                 in_channels: int,
                 patch_size: int,
                 mlp_size: int,
                 n_layers: int,
                 n_heads: int,
                 n_classes: int,
                 atn_dropout: float = 0.0,
                 mlp_dropout: float = 0.1,
                 emb_dropout: float = 0.1):
        super().__init__()

        n_patches = (image_size**2) // (patch_size**2)
        assert image_size % patch_size == 0, "Image size must be divisible by patch size"

        self.cls_embedding = nn.Parameter(
            torch.randn(1, 1, d_model) * 0.02,
            requires_grad=True)
        self.pos_embedding = nn.Parameter(
            torch.randn(1, n_patches+1, d_model) * 0.02,
            requires_grad=True)
        self.emb_dropout = nn.Dropout(emb_dropout)

        self.patcher = PatchEmbedding(d_model, patch_size, in_channels)
        self.encoder = nn.Sequential(*[TransformerEncoder(d_model, n_heads, mlp_size,
                                                          atn_dropout,
                                                          mlp_dropout)
                                       for _ in range(n_layers)])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x: torch.Tensor):
        batch_size, channel, height, width = x.shape
        ## Prepare embeddings
        cls_emb = self.cls_embedding.expand(batch_size, -1, -1)
        pos_emb = self.pos_embedding.expand(batch_size, -1, -1)
        all_emb = self.patcher(x)
        all_emb = torch.cat((cls_emb, all_emb), dim=1) + pos_emb
        all_emb = self.emb_dropout(all_emb)
        # Pass embeddings through the encoder
        output = self.encoder(all_emb)
        # Pass (B, 0, D) through the classifier
        logits = self.classifier(output[:, 0, :])
        return logits

In [ ]:
model = ViT(D, H, C, P, 3072, 12, 12, n_classes).to(device)
model_output = model(torch.randn(1, C, H, W).to(device))
model_output.shape

In [ ]:
summary(model=model,
        input_size=(1, C, H, W),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

## Train Model

In [ ]:
logs_dir, writer = ct.get_summary_writer("pt_vit", "vit_from_scratch", "5_epochs")
logs_dir

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), weight_decay=0.1)
accuracy = MulticlassAccuracy(num_classes=n_classes).to(device)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

In [ ]:
ct.train(model=model,
         tr_dl=tr_dl,
         ts_dl=ts_dl,
         optimizer=optimizer,
         criterion=criterion,
         metric=accuracy,
         n_epochs=N_EPOCHS,
         writer=writer,
         scheduler=scheduler,
         device=device)